# INFERENCE STANDALONE — Metrik Per Skenario dari Dataset yang Sudah Ada

Notebook ini **tidak membuat ulang dataset** dan **tidak membutuhkan file split indices** dari notebook training.

Alur yang dipakai:
1. membaca `manifest.json`;
2. membaca file `.npz` dataset yang sudah dibuat;
3. merekonstruksi pembagian train/validation/test dengan `SEED`, `TRAIN_SIZE`, `VAL_SIZE`, dan `TEST_SIZE` yang sama dengan notebook TRAIN;
4. mengambil bagian test 20% dari setiap skenario;
5. menghitung `accuracy`, `precision`, `recall`, `f1_score`, `Pd`, `Pfa`, dan `youden_index = Pd - Pfa`.

> Catatan: hasil test split akan sama dengan training selama `manifest.json`, urutan file dataset, `SEED`, dan rasio split tidak diubah.


In [1]:
!pip install openpyxl

## 1. Import dan Konfigurasi Awal

In [2]:
import os
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)


TensorFlow: 2.15.0
NumPy: 1.24.4
Pandas: 2.3.3


In [3]:
# Sesuaikan BASE_DIR jika notebook dijalankan dari folder berbeda.
# Struktur default mengikuti notebook pembangkitan dataset:
# BASE_DIR/
#   generated_datasets/
#       manifest.json
#       *.npz
#       best_cnn__all_pu_all_scenarios.keras
#       best_cnn_lstm__all_pu_all_scenarios.keras

BASE_DIR = Path('fading')
OUTPUT_DIR = BASE_DIR / 'generated_datasets'
MANIFEST_PATH = OUTPUT_DIR / 'manifest.json'

CNN_MODEL_PATH = OUTPUT_DIR / 'best_cnn__all_pu_all_scenarios.keras'
CNN_LSTM_MODEL_PATH = OUTPUT_DIR / 'best_cnn_lstm__all_pu_all_scenarios.keras'

# Pilihan mode:
# 'global_test_split' = evaluasi hanya pada test split 20% dari dataset gabungan besar.
# 'full_scenario'     = evaluasi seluruh data pada tiap skenario.
EVALUATION_MODE = 'global_test_split'

# Threshold klasifikasi biner.
THRESHOLD = 0.5

# Proporsi split harus sama dengan notebook training/grid search.
TRAIN_SIZE = 0.7
VAL_SIZE = 0.1
TEST_SIZE = 0.2

print('OUTPUT_DIR:', OUTPUT_DIR.resolve())
print('MANIFEST_PATH:', MANIFEST_PATH.resolve())
print('EVALUATION_MODE:', EVALUATION_MODE)

OUTPUT_DIR: D:\Kampus\SKRIPSI\code\simulasi\finale\fading\generated_datasets
MANIFEST_PATH: D:\Kampus\SKRIPSI\code\simulasi\finale\fading\generated_datasets\manifest.json
EVALUATION_MODE: global_test_split


## Penting: Tidak Perlu Generate Dataset Ulang

Notebook ini dirancang untuk kasus ketika dataset dan model sudah selesai dibuat oleh notebook TRAIN sebelumnya.

- `manifest.json` dipakai untuk mengetahui identitas setiap file: PU berapa, jenis skenario apa, dan nama skenario apa.
- Test split tidak disimpan dari TRAIN lama, sehingga notebook ini merekonstruksi split memakai proses yang sama dengan fungsi `manual_grid_search()` pada TRAIN: `train_test_split(..., random_state=42, stratify=y)`.
- Karena rekonstruksi dilakukan dari urutan gabungan dataset yang sama, bagian test yang dipakai akan konsisten dengan TRAIN lama.
- Notebook ini juga menyimpan file indeks split hasil rekonstruksi agar inference berikutnya tidak perlu merekonstruksi dari awal.


## 2. Helper untuk Membaca Dataset dan Manifest

In [4]:
def load_manifest(manifest_path=MANIFEST_PATH):
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        raise FileNotFoundError(
            f'Manifest tidak ditemukan: {manifest_path}. Jalankan generate_all_datasets() terlebih dahulu.'
        )
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
    return manifest


def load_npz_dataset(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File dataset tidak ditemukan: {path}')
    data = np.load(path, allow_pickle=False)
    X = data['X']
    y = data['y'].astype(int)
    metadata = json.loads(data['metadata'].item())
    return X, y, metadata


def get_dataset_path(item, model_type):
    if model_type == 'cnn':
        return Path(item['cnn_path'])
    if model_type == 'cnn_lstm':
        return Path(item['cnn_lstm_path'])
    raise ValueError("model_type harus 'cnn' atau 'cnn_lstm'")


def make_dataset_table(manifest, model_type):
    rows = []
    for item_index, item in enumerate(manifest):
        dataset_path = get_dataset_path(item, model_type)
        rows.append({
            'item_index': item_index,
            'model_type': model_type,
            'receiver_band': item['receiver_band'],
            'scenario_type': item['scenario_type'],
            'scenario_name': item['scenario_name'],
            'dataset_path': str(dataset_path),
        })
    return pd.DataFrame(rows)


manifest = load_manifest()
print('Jumlah item manifest:', len(manifest))

cnn_table = make_dataset_table(manifest, 'cnn')
cnn_lstm_table = make_dataset_table(manifest, 'cnn_lstm')

display(cnn_table.head())
print('Jumlah file CNN     :', len(cnn_table))
print('Jumlah file CNN-LSTM:', len(cnn_lstm_table))

Jumlah item manifest: 54


,item_index,model_type,receiver_band,scenario_type,scenario_name,dataset_path
0,0,cnn,pu1_low,noise_varying,noise_100_190,fading\generated_datasets\pu1_low__noise_varyi...
1,1,cnn,pu1_low,noise_varying,noise_200_290,fading\generated_datasets\pu1_low__noise_varyi...
2,2,cnn,pu1_low,noise_varying,noise_300_390,fading\generated_datasets\pu1_low__noise_varyi...
3,3,cnn,pu1_low,noise_varying,noise_400_490,fading\generated_datasets\pu1_low__noise_varyi...
4,4,cnn,pu1_low,noise_varying,noise_500_590,fading\generated_datasets\pu1_low__noise_varyi...


Jumlah file CNN     : 54
Jumlah file CNN-LSTM: 54


## 3. Rekonstruksi Global Test Split 70:10:20

Bagian ini dipakai jika `EVALUATION_MODE = 'global_test_split'`. Split direkonstruksi menggunakan urutan file pada `manifest.json`, label `y` dari masing-masing file, dan `random_state` yang sama dengan notebook grid search.

Dengan cara ini, metrik per skenario dihitung hanya dari sampel yang masuk ke **test set global 20%**, bukan dari seluruh data skenario.


In [5]:
def collect_labels_and_ranges(dataset_table):
    y_all_list = []
    ranges = []
    start = 0

    for _, row in dataset_table.iterrows():
        _, y, _ = load_npz_dataset(row['dataset_path'])
        end = start + len(y)
        ranges.append({
            'item_index': int(row['item_index']),
            'start': start,
            'end': end,
            'n_total': len(y),
            'dataset_path': row['dataset_path'],
        })
        y_all_list.append(y)
        start = end

    y_all = np.concatenate(y_all_list, axis=0)
    ranges_df = pd.DataFrame(ranges)
    return y_all, ranges_df


def reconstruct_global_split_indices(y_all, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE, random_state=SEED):
    total_split = train_size + val_size + test_size
    if not np.isclose(total_split, 1.0):
        raise ValueError(f'train_size + val_size + test_size harus 1.0, sekarang {total_split}')

    indices = np.arange(len(y_all))

    idx_train_val, idx_test, y_train_val, y_test = train_test_split(
        indices,
        y_all,
        test_size=test_size,
        stratify=y_all,
        random_state=random_state,
    )

    val_size_relative = val_size / (train_size + val_size)
    idx_train, idx_val, y_train, y_val = train_test_split(
        idx_train_val,
        y_train_val,
        test_size=val_size_relative,
        stratify=y_train_val,
        random_state=random_state,
    )

    return {
        'idx_train': np.sort(idx_train),
        'idx_val': np.sort(idx_val),
        'idx_test': np.sort(idx_test),
    }


def local_test_indices_for_file(global_test_indices, file_start, file_end):
    mask = (global_test_indices >= file_start) & (global_test_indices < file_end)
    return global_test_indices[mask] - file_start


def save_reconstructed_split_indices(split_indices, model_type):
    split_path = OUTPUT_DIR / f'reconstructed_global_split_indices_{model_type}.npz'
    np.savez_compressed(
        split_path,
        idx_train=split_indices['idx_train'],
        idx_val=split_indices['idx_val'],
        idx_test=split_indices['idx_test'],
        seed=SEED,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
    print('Split indices rekonstruksi disimpan ke:', split_path)


def build_split_context(dataset_table, model_type):
    y_all, ranges_df = collect_labels_and_ranges(dataset_table)
    split_indices = reconstruct_global_split_indices(y_all)
    print('Total data:', len(y_all))
    print('Train     :', len(split_indices['idx_train']))
    print('Val       :', len(split_indices['idx_val']))
    print('Test      :', len(split_indices['idx_test']))
    save_reconstructed_split_indices(split_indices, model_type)
    return y_all, ranges_df, split_indices


# Rekonstruksi split bersifat per model_type karena urutan file CNN dan CNN-LSTM dibaca terpisah.
if EVALUATION_MODE == 'global_test_split':
    print('Membangun split context CNN...')
    y_cnn_all, cnn_ranges_df, cnn_split = build_split_context(cnn_table, 'cnn')

    print('')
    print('Membangun split context CNN-LSTM...')
    y_lstm_all, lstm_ranges_df, lstm_split = build_split_context(cnn_lstm_table, 'cnn_lstm')
else:
    cnn_ranges_df = None
    lstm_ranges_df = None
    cnn_split = None
    lstm_split = None


Membangun split context CNN...
Total data: 108000
Train     : 75599
Val       : 10801
Test      : 21600
Split indices rekonstruksi disimpan ke: fading\generated_datasets\reconstructed_global_split_indices_cnn.npz

Membangun split context CNN-LSTM...
Total data: 108000
Train     : 75599
Val       : 10801
Test      : 21600
Split indices rekonstruksi disimpan ke: fading\generated_datasets\reconstructed_global_split_indices_cnn_lstm.npz


## 4. Helper Metrik Inference

Definisi label:

- `1`: PU hadir / PU + noise
- `0`: PU tidak hadir / noise only

Maka:

- `Pd = TP / (TP + FN)`
- `Pfa = FP / (FP + TN)`
- `Youden Index = Pd - Pfa`


In [6]:
def predict_binary(model, X, threshold=THRESHOLD, batch_size=256):
    raw = model.predict(X, batch_size=batch_size, verbose=0)

    # Model training memakai Dense(2, activation='softmax').
    # Untuk format ini, probabilitas kelas PU hadir adalah kolom ke-1.
    # Jika suatu saat model diganti menjadi sigmoid 1 output, fungsi ini tetap aman.
    if raw.ndim == 2 and raw.shape[1] == 2:
        proba = raw[:, 1]
    elif raw.ndim == 2 and raw.shape[1] == 1:
        proba = raw[:, 0]
    elif raw.ndim == 1:
        proba = raw
    else:
        raise ValueError(f'Bentuk output model tidak dikenali: {raw.shape}')

    y_pred = (proba >= threshold).astype(int)
    return y_pred, proba


def compute_binary_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    pd_value = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pfa_value = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    youden_index = pd_value - pfa_value

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'pd': pd_value,
        'pfa': pfa_value,
        'youden_index': youden_index,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'support_0_noise_only': int(tn + fp),
        'support_1_pu_noise': int(tp + fn),
    }


def evaluate_one_dataset_file(
    model,
    row,
    evaluation_mode=EVALUATION_MODE,
    ranges_df=None,
    split_indices=None,
    threshold=THRESHOLD,
):
    X, y, metadata = load_npz_dataset(row['dataset_path'])

    if evaluation_mode == 'full_scenario':
        eval_indices = np.arange(len(y))
    elif evaluation_mode == 'global_test_split':
        if ranges_df is None or split_indices is None:
            raise ValueError('ranges_df dan split_indices wajib tersedia untuk global_test_split.')

        range_row = ranges_df.loc[ranges_df['item_index'] == int(row['item_index'])].iloc[0]
        eval_indices = local_test_indices_for_file(
            split_indices['idx_test'],
            int(range_row['start']),
            int(range_row['end']),
        )
    else:
        raise ValueError("evaluation_mode harus 'global_test_split' atau 'full_scenario'")

    if len(eval_indices) == 0:
        raise ValueError(f'Tidak ada data evaluasi untuk file: {row["dataset_path"]}')

    X_eval = X[eval_indices]
    y_eval = y[eval_indices]
    y_pred, proba = predict_binary(model, X_eval, threshold=threshold)
    metrics = compute_binary_metrics(y_eval, y_pred)

    result = {
        'model_type': row['model_type'],
        'receiver_band': row['receiver_band'],
        'scenario_type': row['scenario_type'],
        'scenario_name': row['scenario_name'],
        'evaluation_mode': evaluation_mode,
        'threshold': threshold,
        'n_total_file': int(len(y)),
        'n_eval': int(len(y_eval)),
        'dataset_path': row['dataset_path'],
    }
    result.update(metrics)
    return result


def evaluate_model_per_scenario(
    model_path,
    dataset_table,
    model_type,
    evaluation_mode=EVALUATION_MODE,
    ranges_df=None,
    split_indices=None,
    threshold=THRESHOLD,
):
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f'Model tidak ditemukan: {model_path}')

    print(f'Load model {model_type}: {model_path}')
    model = tf.keras.models.load_model(model_path)

    rows = []
    for i, (_, row) in enumerate(dataset_table.iterrows(), start=1):
        print(f'[{i}/{len(dataset_table)}] {model_type} | {row["receiver_band"]} | {row["scenario_type"]} | {row["scenario_name"]}')
        result = evaluate_one_dataset_file(
            model=model,
            row=row,
            evaluation_mode=evaluation_mode,
            ranges_df=ranges_df,
            split_indices=split_indices,
            threshold=threshold,
        )
        result['model_path'] = str(model_path)
        rows.append(result)

    return pd.DataFrame(rows)

## 5. Jalankan Inference per Skenario

Cell berikut sudah dibuat untuk langsung menjalankan inference CNN dan CNN-LSTM saat notebook dijalankan dari atas ke bawah.

Pastikan file berikut sudah ada dari notebook training:

```text
best_cnn__all_pu_all_scenarios.keras
best_cnn_lstm__all_pu_all_scenarios.keras
manifest.json
```


In [7]:
# FINAL: dibuat True agar notebook inference bisa dijalankan sekali dari atas ke bawah.
RUN_INFERENCE_CNN = True
RUN_INFERENCE_CNN_LSTM = True

all_results = []

if RUN_INFERENCE_CNN:
    cnn_results = evaluate_model_per_scenario(
        model_path=CNN_MODEL_PATH,
        dataset_table=cnn_table,
        model_type='cnn',
        evaluation_mode=EVALUATION_MODE,
        ranges_df=cnn_ranges_df,
        split_indices=cnn_split,
        threshold=THRESHOLD,
    )
    all_results.append(cnn_results)
    display(cnn_results.head())
    gc.collect()
    tf.keras.backend.clear_session()

if RUN_INFERENCE_CNN_LSTM:
    cnn_lstm_results = evaluate_model_per_scenario(
        model_path=CNN_LSTM_MODEL_PATH,
        dataset_table=cnn_lstm_table,
        model_type='cnn_lstm',
        evaluation_mode=EVALUATION_MODE,
        ranges_df=lstm_ranges_df,
        split_indices=lstm_split,
        threshold=THRESHOLD,
    )
    all_results.append(cnn_lstm_results)
    display(cnn_lstm_results.head())
    gc.collect()
    tf.keras.backend.clear_session()

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    display(results_df)
else:
    print('Inference belum dijalankan. Ubah RUN_INFERENCE_CNN atau RUN_INFERENCE_CNN_LSTM menjadi True.')


Load model cnn: fading\generated_datasets\best_cnn__all_pu_all_scenarios.keras


[1/54] cnn | pu1_low | noise_varying | noise_100_190
[2/54] cnn | pu1_low | noise_varying | noise_200_290
[3/54] cnn | pu1_low | noise_varying | noise_300_390
[4/54] cnn | pu1_low | noise_varying | noise_400_490
[5/54] cnn | pu1_low | noise_varying | noise_500_590
[6/54] cnn | pu1_low | noise_varying | noise_600_690
[7/54] cnn | pu1_low | fading_varying | fading-10
[8/54] cnn | pu1_low | fading_varying | fading-11
[9/54] cnn | pu1_low | fading_varying | fading-12
[10/54] cnn | pu1_low | fading_varying | fading-13
[11/54] cnn | pu1_low | fading_varying | fading-14
[12/54] cnn | pu1_low | fading_varying | fading-15
[13/54] cnn | pu1_low | noise_and_fading_varying | fading-10
[14/54] cnn | pu1_low | noise_and_fading_varying | fading-11
[15/54] cnn | pu1_low | noise_and_fading_varying | fading-12
[16/54] cnn | pu1_low | noise_and_fading_varying | fading-13
[17/54] cnn | pu1_low | noise_and_fading_varying | fad

,model_type,receiver_band,scenario_type,scenario_name,evaluation_mode,threshold,n_total_file,n_eval,dataset_path,accuracy,...,pd,pfa,youden_index,tn,fp,fn,tp,support_0_noise_only,support_1_pu_noise,model_path
0,cnn,pu1_low,noise_varying,noise_100_190,global_test_split,0.5,2000,423,fading\generated_datasets\pu1_low__noise_varyi...,1.000000,...,1.000000,0.000000,1.000000,202,0,0,221,202,221,fading\generated_datasets\best_cnn__all_pu_all...
1,cnn,pu1_low,noise_varying,noise_200_290,global_test_split,0.5,2000,385,fading\generated_datasets\pu1_low__noise_varyi...,0.984416,...,0.976879,0.009434,0.967445,210,2,4,169,212,173,fading\generated_datasets\best_cnn__all_pu_all...
2,cnn,pu1_low,noise_varying,noise_300_390,global_test_split,0.5,2000,411,fading\generated_datasets\pu1_low__noise_varyi...,0.890511,...,0.841860,0.056122,0.785738,185,11,34,181,196,215,fading\generated_datasets\best_cnn__all_pu_all...
3,cnn,pu1_low,noise_varying,noise_400_490,global_test_split,0.5,2000,401,fading\generated_datasets\pu1_low__noise_varyi...,0.835411,...,0.715000,0.044776,0.670224,192,9,57,143,201,200,fading\generated_datasets\best_cnn__all_pu_all...
4,cnn,pu1_low,noise_varying,noise_500_590,global_test_split,0.5,2000,407,fading\generated_datasets\pu1_low__noise_varyi...,0.803440,...,0.696078,0.088670,0.607408,185,18,62,142,203,204,fading\generated_datasets\best_cnn__all_pu_all...


Load model cnn_lstm: fading\generated_datasets\best_cnn_lstm__all_pu_all_scenarios.keras
[1/54] cnn_lstm | pu1_low | noise_varying | noise_100_190
[2/54] cnn_lstm | pu1_low | noise_varying | noise_200_290
[3/54] cnn_lstm | pu1_low | noise_varying | noise_300_390
[4/54] cnn_lstm | pu1_low | noise_varying | noise_400_490
[5/54] cnn_lstm | pu1_low | noise_varying | noise_500_590
[6/54] cnn_lstm | pu1_low | noise_varying | noise_600_690
[7/54] cnn_lstm | pu1_low | fading_varying | fading-10
[8/54] cnn_lstm | pu1_low | fading_varying | fading-11
[9/54] cnn_lstm | pu1_low | fading_varying | fading-12
[10/54] cnn_lstm | pu1_low | fading_varying | fading-13
[11/54] cnn_lstm | pu1_low | fading_varying | fading-14
[12/54] cnn_lstm | pu1_low | fading_varying | fading-15
[13/54] cnn_lstm | pu1_low | noise_and_fading_varying | fading-10
[14/54] cnn_lstm | pu1_low | noise_and_fading_varying | fading-11
[15/54] cnn_lstm | pu1_low | noise_and_fading_varying | fading-12
[16/54] cnn_lstm | pu1_low | noi

,model_type,receiver_band,scenario_type,scenario_name,evaluation_mode,threshold,n_total_file,n_eval,dataset_path,accuracy,...,pd,pfa,youden_index,tn,fp,fn,tp,support_0_noise_only,support_1_pu_noise,model_path
0,cnn_lstm,pu1_low,noise_varying,noise_100_190,global_test_split,0.5,2000,423,fading\generated_datasets\pu1_low__noise_varyi...,0.973995,...,1.000000,0.054455,0.945545,191,11,0,221,202,221,fading\generated_datasets\best_cnn_lstm__all_p...
1,cnn_lstm,pu1_low,noise_varying,noise_200_290,global_test_split,0.5,2000,385,fading\generated_datasets\pu1_low__noise_varyi...,0.958442,...,0.930636,0.018868,0.911768,208,4,12,161,212,173,fading\generated_datasets\best_cnn_lstm__all_p...
2,cnn_lstm,pu1_low,noise_varying,noise_300_390,global_test_split,0.5,2000,411,fading\generated_datasets\pu1_low__noise_varyi...,0.802920,...,0.683721,0.066327,0.617394,183,13,68,147,196,215,fading\generated_datasets\best_cnn_lstm__all_p...
3,cnn_lstm,pu1_low,noise_varying,noise_400_490,global_test_split,0.5,2000,401,fading\generated_datasets\pu1_low__noise_varyi...,0.770574,...,0.580000,0.039801,0.540199,193,8,84,116,201,200,fading\generated_datasets\best_cnn_lstm__all_p...
4,cnn_lstm,pu1_low,noise_varying,noise_500_590,global_test_split,0.5,2000,407,fading\generated_datasets\pu1_low__noise_varyi...,0.729730,...,0.500000,0.039409,0.460591,195,8,102,102,203,204,fading\generated_datasets\best_cnn_lstm__all_p...


,model_type,receiver_band,scenario_type,scenario_name,evaluation_mode,threshold,n_total_file,n_eval,dataset_path,accuracy,...,pd,pfa,youden_index,tn,fp,fn,tp,support_0_noise_only,support_1_pu_noise,model_path
0,cnn,pu1_low,noise_varying,noise_100_190,global_test_split,0.5,2000,423,fading\generated_datasets\pu1_low__noise_varyi...,1.000000,...,1.000000,0.000000,1.000000,202,0,0,221,202,221,fading\generated_datasets\best_cnn__all_pu_all...
1,cnn,pu1_low,noise_varying,noise_200_290,global_test_split,0.5,2000,385,fading\generated_datasets\pu1_low__noise_varyi...,0.984416,...,0.976879,0.009434,0.967445,210,2,4,169,212,173,fading\generated_datasets\best_cnn__all_pu_all...
2,cnn,pu1_low,noise_varying,noise_300_390,global_test_split,0.5,2000,411,fading\generated_datasets\pu1_low__noise_varyi...,0.890511,...,0.841860,0.056122,0.785738,185,11,34,181,196,215,fading\generated_datasets\best_cnn__all_pu_all...
3,cnn,pu1_low,noise_varying,noise_400_490,global_test_split,0.5,2000,401,fading\generated_datasets\pu1_low__noise_varyi...,0.835411,...,0.715000,0.044776,0.670224,192,9,57,143,201,200,fading\generated_datasets\best_cnn__all_pu_all...
4,cnn,pu1_low,noise_varying,noise_500_590,global_test_split,0.5,2000,407,fading\generated_datasets\pu1_low__noise_varyi...,0.803440,...,0.696078,0.088670,0.607408,185,18,62,142,203,204,fading\generated_datasets\best_cnn__all_pu_all...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,cnn_lstm,pu3_high,noise_and_fading_varying,fading-11,global_test_split,0.5,2000,411,fading\generated_datasets\pu3_high__noise_and_...,0.452555,...,0.037383,0.096447,-0.059064,178,19,206,8,197,214,fading\generated_datasets\best_cnn_lstm__all_p...
104,cnn_lstm,pu3_high,noise_and_fading_varying,fading-12,global_test_split,0.5,2000,430,fading\generated_datasets\pu3_high__noise_and_...,0.453488,...,0.030568,0.064677,-0.034109,188,13,222,7,201,229,fading\generated_datasets\best_cnn_lstm__all_p...
105,cnn_lstm,pu3_high,noise_and_fading_varying,fading-13,global_test_split,0.5,2000,406,fading\generated_datasets\pu3_high__noise_and_...,0.544335,...,0.162679,0.050761,0.111918,187,10,175,34,197,209,fading\generated_datasets\best_cnn_lstm__all_p...
106,cnn_lstm,pu3_high,noise_and_fading_varying,fading-14,global_test_split,0.5,2000,401,fading\generated_datasets\pu3_high__noise_and_...,0.511222,...,0.098446,0.105769,-0.007324,186,22,174,19,208,193,fading\generated_datasets\best_cnn_lstm__all_p...


## 6. Simpan Hasil Inference

In [8]:
if 'results_df' in globals() and not results_df.empty:
    inference_csv_path = OUTPUT_DIR / f'inference_metrics_per_scenario__{EVALUATION_MODE}.csv'
    inference_xlsx_path = OUTPUT_DIR / f'inference_metrics_per_scenario__{EVALUATION_MODE}.xlsx'

    results_df.to_csv(inference_csv_path, index=False)
    print('CSV disimpan ke :', inference_csv_path)

    try:
        results_df.to_excel(inference_xlsx_path, index=False)
        print('Excel disimpan ke:', inference_xlsx_path)
    except Exception as err:
        print('Excel tidak berhasil disimpan. CSV tetap tersedia.')
        print('Error:', err)
else:
    print('Belum ada results_df. Jalankan inference terlebih dahulu.')


CSV disimpan ke : fading\generated_datasets\inference_metrics_per_scenario__global_test_split.csv
Excel disimpan ke: fading\generated_datasets\inference_metrics_per_scenario__global_test_split.xlsx


## 7. Rekap Rata-rata per Kelompok

Bagian ini opsional, tetapi berguna untuk melihat performa rata-rata per PU, per jenis skenario, atau per model.


In [9]:
METRIC_COLUMNS = ['accuracy', 'precision', 'recall', 'f1_score', 'pd', 'pfa', 'youden_index']

if 'results_df' in globals() and not results_df.empty:
    summary_by_model = results_df.groupby('model_type')[METRIC_COLUMNS].mean().reset_index()
    summary_by_pu = results_df.groupby(['model_type', 'receiver_band'])[METRIC_COLUMNS].mean().reset_index()
    summary_by_type = results_df.groupby(['model_type', 'scenario_type'])[METRIC_COLUMNS].mean().reset_index()
    summary_by_pu_type = results_df.groupby(['model_type', 'receiver_band', 'scenario_type'])[METRIC_COLUMNS].mean().reset_index()

    print('Rata-rata per model')
    display(summary_by_model)

    print('Rata-rata per PU')
    display(summary_by_pu)

    print('Rata-rata per jenis skenario')
    display(summary_by_type)

    print('Rata-rata per PU dan jenis skenario')
    display(summary_by_pu_type)
else:
    print('Belum ada results_df. Jalankan inference terlebih dahulu.')

Rata-rata per model


,model_type,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,0.871641,0.886442,0.850765,0.864133,0.850765,0.108454,0.742311
1,cnn_lstm,0.633112,0.708638,0.362635,0.448363,0.362635,0.095115,0.267519


Rata-rata per PU


,model_type,receiver_band,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,pu1_low,0.868770,0.889984,0.837929,0.859416,0.837929,0.100875,0.737054
1,cnn,pu2_mid,0.874638,0.888641,0.850537,0.866351,0.850537,0.102672,0.747865
2,cnn,pu3_high,0.871515,0.880700,0.863829,0.866631,0.863829,0.121816,0.742013
3,cnn_lstm,pu1_low,0.639173,0.736200,0.362352,0.459234,0.362352,0.088112,0.274241
4,cnn_lstm,pu2_mid,0.620493,0.686069,0.378148,0.458291,0.378148,0.135740,0.242409
5,cnn_lstm,pu3_high,0.639670,0.703645,0.347403,0.427564,0.347403,0.061495,0.285908


Rata-rata per jenis skenario


,model_type,scenario_type,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,fading_varying,0.908115,0.909871,0.903721,0.903968,0.903721,0.089090,0.814631
1,cnn,noise_and_fading_varying,0.816980,0.813407,0.814715,0.809906,0.814715,0.183000,0.631714
2,cnn,noise_varying,0.889828,0.936047,0.833860,0.878524,0.833860,0.053272,0.780588
3,cnn_lstm,fading_varying,0.568669,0.662597,0.212203,0.305499,0.212203,0.073745,0.138457
4,cnn_lstm,noise_and_fading_varying,0.521175,0.553447,0.194507,0.278375,0.194507,0.150172,0.044335
5,cnn_lstm,noise_varying,0.809492,0.909869,0.681194,0.761214,0.681194,0.061429,0.619765


Rata-rata per PU dan jenis skenario


,model_type,receiver_band,scenario_type,accuracy,precision,recall,f1_score,pd,pfa,youden_index
0,cnn,pu1_low,fading_varying,0.907501,0.912713,0.894324,0.901216,0.894324,0.081571,0.812753
1,cnn,pu1_low,noise_and_fading_varying,0.812939,0.815174,0.800712,0.803940,0.800712,0.174396,0.626317
2,cnn,pu1_low,noise_varying,0.885869,0.942064,0.818752,0.873092,0.818752,0.046659,0.772092
3,cnn,pu2_mid,fading_varying,0.905607,0.918182,0.891762,0.901171,0.891762,0.080896,0.810865
4,cnn,pu2_mid,noise_and_fading_varying,0.816142,0.829668,0.781610,0.800778,0.781610,0.153495,0.628114
5,cnn,pu2_mid,noise_varying,0.902166,0.918073,0.878239,0.897104,0.878239,0.073623,0.804616
6,cnn,pu3_high,fading_varying,0.911238,0.898718,0.925078,0.909517,0.925078,0.104804,0.820274
7,cnn,pu3_high,noise_and_fading_varying,0.821859,0.795381,0.861822,0.825001,0.861822,0.221110,0.640712
8,cnn,pu3_high,noise_varying,0.881450,0.948002,0.804588,0.865376,0.804588,0.039533,0.765055
9,cnn_lstm,pu1_low,fading_varying,0.578734,0.722409,0.222953,0.334576,0.222953,0.073879,0.149073


In [10]:
if 'results_df' in globals() and not results_df.empty:
    summary_by_model.to_csv(OUTPUT_DIR / f'inference_summary_by_model__{EVALUATION_MODE}.csv', index=False)
    summary_by_pu.to_csv(OUTPUT_DIR / f'inference_summary_by_pu__{EVALUATION_MODE}.csv', index=False)
    summary_by_type.to_csv(OUTPUT_DIR / f'inference_summary_by_scenario_type__{EVALUATION_MODE}.csv', index=False)
    summary_by_pu_type.to_csv(OUTPUT_DIR / f'inference_summary_by_pu_and_scenario_type__{EVALUATION_MODE}.csv', index=False)

    summary_xlsx_path = OUTPUT_DIR / f'inference_metrics_and_summaries__{EVALUATION_MODE}.xlsx'
    try:
        with pd.ExcelWriter(summary_xlsx_path) as writer:
            results_df.to_excel(writer, sheet_name='per_scenario', index=False)
            summary_by_model.to_excel(writer, sheet_name='summary_by_model', index=False)
            summary_by_pu.to_excel(writer, sheet_name='summary_by_pu', index=False)
            summary_by_type.to_excel(writer, sheet_name='summary_by_type', index=False)
            summary_by_pu_type.to_excel(writer, sheet_name='summary_by_pu_type', index=False)
        print('Excel gabungan disimpan ke:', summary_xlsx_path)
    except Exception as err:
        print('Excel gabungan tidak berhasil disimpan. Seluruh CSV summary tetap tersedia.')
        print('Error:', err)

    print('Seluruh file summary CSV sudah disimpan di:', OUTPUT_DIR)
else:
    print('Belum ada results_df. Jalankan inference terlebih dahulu.')


Excel gabungan disimpan ke: fading\generated_datasets\inference_metrics_and_summaries__global_test_split.xlsx
Seluruh file summary CSV sudah disimpan di: fading\generated_datasets


## 8. Catatan Interpretasi

Untuk pelaporan skripsi, gunakan mode `global_test_split` agar metrik per skenario hanya dihitung dari test set 20%. Mode `full_scenario` dapat dipakai sebagai analisis tambahan, tetapi perlu diberi catatan karena data yang dievaluasi dapat mencakup data yang pernah masuk proses training/validation pada grid search.

Interpretasi metrik:

- `accuracy`: proporsi seluruh prediksi yang benar.
- `precision`: dari semua prediksi PU hadir, berapa yang benar-benar PU hadir.
- `recall`: sama dengan sensitivitas untuk kelas PU hadir.
- `f1_score`: harmonisasi precision dan recall.
- `pd`: probability of detection, sama dengan `TP / (TP + FN)`.
- `pfa`: probability of false alarm, sama dengan `FP / (FP + TN)`.
- `youden_index`: selisih `pd - pfa`; makin mendekati 1 berarti deteksi semakin baik dengan false alarm rendah.

Output utama:

```text
generated_datasets/
├── inference_metrics_per_scenario__global_test_split.csv
├── inference_metrics_per_scenario__global_test_split.xlsx
├── inference_summary_by_model__global_test_split.csv
├── inference_summary_by_pu__global_test_split.csv
├── inference_summary_by_scenario_type__global_test_split.csv
├── inference_summary_by_pu_and_scenario_type__global_test_split.csv
└── inference_metrics_and_summaries__global_test_split.xlsx
```
